# HPPSO — 20 Shifted Benchmarks: Analysis & Reproduction (Cleaned)

## Purpose
Refactored version of `original/HPPSO_Final_Reviiew.ipynb`. This notebook **does not re-run** the full benchmark suite. It loads existing pickles, merges CMA-ES results, and reproduces paper tables/figures.

## Original notebook scope (preserved logic)
- 20 shifted functions at **30D** and **1000D**
- 20 independent runs per setting
- Algorithms: PSO, PSO-M, PSO-RIW, HPPSO, GA-MPC, GWO, SHADE, CSA, CMA-ES
- Wilcoxon pairwise scoring, average ranks, convergence plots

## Result files required (`results/`)
| File | Content |
|------|---------|
| `all_convergence_histories 30d.pkl` | Non-CMA 30D histories |
| `all_convergence_histories 30d cmaes only.pkl` | CMA-ES 30D histories |
| `all_convergence_histories1000.pkl` | 1000D histories |
| `best_costs_30d.pkl` | 30D final costs |
| `all_svs30d.pkl`, `all_svs 1000.pkl` | Shift vectors |

## Paper artifacts produced
- **Tables 2–7**, **Figures 7–8**
- Exports under `reproduced_tables/`, `reproduced_figures/`


## 1. Imports and paths


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate repo root from notebooks/ or notebooks/original/
_cwd = Path.cwd()
if (_cwd / "nb_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
    REPO_ROOT = _cwd.parent
elif (_cwd.parent / "nb_helpers.py").exists():
    sys.path.insert(0, str(_cwd.parent))
    REPO_ROOT = _cwd.parent.parent
else:
    REPO_ROOT = _cwd

import nb_helpers as nh
nh.ensure_reproduction_imports()

from reproduction.merge import merge_dimension
from reproduction.aggregate import (
    summarize_performance,
    average_ranks,
    wilcoxon_pairwise_scores,
    median_convergence_curve,
    internal_to_paper,
)
from reproduction.config import FUNCTION_NAMES, FIGURE_PANEL_FUNCTIONS, HPPSO_ALGORITHM_KEY
from reproduction.tables import save_tables
from reproduction.visualize import (
    plot_hppso_median_panels,
    plot_all_algorithms_convergence,
    plot_average_rank_bar,
)
from reproduction.export_shift_vectors import export_all_shift_vectors

plt.rcParams.update(nh.PLOT_STYLE)

RESULTS_DIR = nh.RESULTS_DIR
OUT_TABLES = nh.REPRODUCED_TABLES
OUT_FIGURES = nh.REPRODUCED_FIGURES
OUT_TABLES.mkdir(exist_ok=True)
OUT_FIGURES.mkdir(exist_ok=True)


## 2. Merge policy (CMA-ES)

30D: embedded CMA-ES entries in the main pickle are **replaced** by `all_convergence_histories 30d cmaes only.pkl`.

1000D: single file already contains all algorithms.

Dimensions are never mixed.


## 3. Load and merge datasets


In [ ]:
datasets = {}
merge_reports = {}
for dim in (30, 1000):
    ds = merge_dimension(dim)
    datasets[dim] = ds
    merge_reports[dim] = ds.merge_report
    print(f"\n{dim}D: {len(ds.algorithms())} algorithms, {len(ds.functions())} functions")
    print(f"  CMA-ES from separate file: {ds.merge_report.replaced_cmaes_from_separate_file}")
    if ds.merge_report.warnings:
        for w in ds.merge_report.warnings[:3]:
            print(f"  note: {w}")


## 4. Performance summaries (Tables 2 & 3)


In [ ]:
for dim, ds in datasets.items():
    perf = summarize_performance(ds)
    print(f"\n=== {dim}D mean best cost (first 5 rows) ===")
    display(perf.head())
    pivot = perf.pivot(index="function", columns="algorithm", values="mean")
    display(pivot.iloc[:5, :5])


## 5. Average ranks (Tables 4 & 5)


In [ ]:
for dim, ds in datasets.items():
    ranks = average_ranks(ds)
    print(f"\n{dim}D average ranks:")
    display(ranks)
    rank_fig = plot_average_rank_bar(ds, ranks, output_dir=OUT_FIGURES)
    print(f"  saved {rank_fig.name}")


## 6. Wilcoxon pairwise scores (Tables 6 & 7)

Score = (wins + 0.5 × ties) / total_comparisons × 100, comparing pooled run-level costs across all functions.


In [ ]:
for dim, ds in datasets.items():
    exclude = {"f2_schwefel_2_22"} if dim == 1000 else set()
    wdf = wilcoxon_pairwise_scores(ds, exclude_functions=exclude)
    print(f"\n{dim}D Wilcoxon ranking:")
    display(wdf)


## 7. Export CSV tables


In [ ]:
for dim, ds in datasets.items():
    paths = save_tables(ds, output_dir=OUT_TABLES)
    print(f"{dim}D:", ", ".join(p.name for p in paths.values()))


## 8. Shift vectors


In [ ]:
from pathlib import Path

manifest = export_all_shift_vectors(REPO_ROOT)
for dim, meta in manifest.items():
    print(f"{dim}D shift vectors -> {Path(meta['exported_csv']).name}")


## 9. Figures 7 & 8 — HPPSO median convergence (15 panels each)


In [ ]:
for dim in (30, 1000):
    fig_num = 7 if dim == 30 else 8
    path = plot_hppso_median_panels(datasets[dim], figure_number=fig_num, output_dir=OUT_FIGURES)
    print(f"Figure {fig_num} ({dim}D): {path}")


## 10. Supplementary — all-algorithm convergence (selected functions)


In [ ]:
example_functions = ["f1_sphere", "f6_rastrigin", "f7_ackley"]
for dim in (30, 1000):
    for func in example_functions:
        p = plot_all_algorithms_convergence(
            datasets[dim], func, output_dir=OUT_FIGURES / "supplementary"
        )
        if p:
            print(f"  {dim}D {func}: {p.name}")


## 11. Export long-format merged CSV


In [ ]:
for dim, ds in datasets.items():
    csv_path = REPO_ROOT / f"merged_results_{dim}D.csv"
    ds.to_long_dataframe().to_csv(csv_path, index=False)
    print(f"Saved {csv_path.name}")


## Removed from original (exploratory / duplicate)
- Inline redefinitions of PSO, GWO, SHADE, CMA-ES (now in `src/hppso/`)
- 2000D sphere demo runs
- Parameter sensitivity sweeps
- Duplicate empty cells and repeated CSV exports

## Full automation
Run `python -m reproduction.run_reproduction` for the same pipeline without Jupyter.
